# PySpark CDC — Audit, Idempotency and Deduplication

CDC pipelines need deterministic ordering and deduplication. This notebook shows how to remove duplicate events and keep a small audit view.

## 00 — Spark setup and sample data

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("cdc-delta-cdf")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/cdc-delta-cdf_warehouse")
    .getOrCreate()
)

spark.version

'4.0.2'

In [2]:
cdc_events = spark.createDataFrame([
    ("evt-001", 1, "Alice", "alice@new.com", "U", "2026-05-02T10:05:00"),
    ("evt-001", 1, "Alice", "alice@new.com", "U", "2026-05-02T10:05:00"),
    ("evt-002", 2, None, None, "D", "2026-05-02T10:10:00"),
    ("evt-003", 3, "Carol", "carol@mail.com", "I", "2026-05-02T10:15:00"),
    ("evt-004", 3, "Carol", "carol.new@mail.com", "U", "2026-05-02T10:20:00"),
], ["event_id", "customer_id", "name", "email", "op", "event_time"]).withColumn(
    "event_time",
    F.to_timestamp("event_time")
)

cdc_events.orderBy("event_time", "event_id").show(truncate=False)

[Stage 0:>                                                          (0 + 2) / 2]

+--------+-----------+-----+------------------+---+-------------------+
|event_id|customer_id|name |email             |op |event_time         |
+--------+-----------+-----+------------------+---+-------------------+
|evt-001 |1          |Alice|alice@new.com     |U  |2026-05-02 10:05:00|
|evt-001 |1          |Alice|alice@new.com     |U  |2026-05-02 10:05:00|
|evt-002 |2          |NULL |NULL              |D  |2026-05-02 10:10:00|
|evt-003 |3          |Carol|carol@mail.com    |I  |2026-05-02 10:15:00|
|evt-004 |3          |Carol|carol.new@mail.com|U  |2026-05-02 10:20:00|
+--------+-----------+-----+------------------+---+-------------------+



## 01 — Remove exact duplicate CDC events

In [3]:
deduplicated_events = cdc_events.dropDuplicates(["event_id"])

deduplicated_events.orderBy("event_time", "event_id").show(truncate=False)

[Stage 1:>                                                          (0 + 2) / 2]

+--------+-----------+-----+------------------+---+-------------------+
|event_id|customer_id|name |email             |op |event_time         |
+--------+-----------+-----+------------------+---+-------------------+
|evt-001 |1          |Alice|alice@new.com     |U  |2026-05-02 10:05:00|
|evt-002 |2          |NULL |NULL              |D  |2026-05-02 10:10:00|
|evt-003 |3          |Carol|carol@mail.com    |I  |2026-05-02 10:15:00|
|evt-004 |3          |Carol|carol.new@mail.com|U  |2026-05-02 10:20:00|
+--------+-----------+-----+------------------+---+-------------------+



## 02 — Build an audit summary

In [4]:
audit_summary = (
    deduplicated_events
    .groupBy("op")
    .agg(
        F.count("*").alias("event_count"),
        F.min("event_time").alias("first_event_time"),
        F.max("event_time").alias("last_event_time")
    )
    .orderBy("op")
)

audit_summary.show(truncate=False)

+---+-----------+-------------------+-------------------+
|op |event_count|first_event_time   |last_event_time    |
+---+-----------+-------------------+-------------------+
|D  |1          |2026-05-02 10:10:00|2026-05-02 10:10:00|
|I  |1          |2026-05-02 10:15:00|2026-05-02 10:15:00|
|U  |2          |2026-05-02 10:05:00|2026-05-02 10:20:00|
+---+-----------+-------------------+-------------------+



## 03 — Latest event per key after deduplication

In [5]:
from pyspark.sql.window import Window

latest_window = Window.partitionBy("customer_id").orderBy(F.col("event_time").desc())

latest_cdc_per_key = (
    deduplicated_events
    .withColumn("rn", F.row_number().over(latest_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

latest_cdc_per_key.orderBy("customer_id").show(truncate=False)

+--------+-----------+-----+------------------+---+-------------------+
|event_id|customer_id|name |email             |op |event_time         |
+--------+-----------+-----+------------------+---+-------------------+
|evt-001 |1          |Alice|alice@new.com     |U  |2026-05-02 10:05:00|
|evt-002 |2          |NULL |NULL              |D  |2026-05-02 10:10:00|
|evt-004 |3          |Carol|carol.new@mail.com|U  |2026-05-02 10:20:00|
+--------+-----------+-----+------------------+---+-------------------+

